In [14]:
!gdown 10bnEC6-ZfXZFZ2mb3zoWd38TjYufanWo

^C


In [1]:
import os
import random

import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm 

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models, datasets
import torch.nn.functional as F
from torch import nn, optim

from sklearn.metrics import log_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# 시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
import torch
from torchvision import datasets
from torchvision.transforms import v2 
from torch.utils.data import DataLoader, random_split

# 1. v2 전처리 파이프라인 정의
data_transforms = v2.Compose([
    v2.ToImage(),                          # 1) PIL 이미지를 v2 Image 객체(텐서)로 변환
    v2.ToDtype(torch.float32, scale=True), # 2) float32 타입 변환 및 0~1 정규화 (기존 ToTensor 역할)
    v2.Resize(size=(224, 224)),            # 3) 고속 이미지 리사이즈
    v2.Normalize(mean=[0.485, 0.456, 0.406], 
                 std=[0.229, 0.224, 0.225]) # 4) ImageNet 표준 정규화
])

# 2. ImageFolder를 이용한 데이터셋 로드
data_dir = "./Pistachio_Image_Dataset" 
dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)

# 3. 학습용(80%) / 검증용(20%) 데이터 분할
generator = torch.Generator().manual_seed(42) # 결과 재현을 위한 랜덤 시드 고정
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

# 4. DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# 5. 로드 결과 및 첫 번째 배치의 텐서 타입 검증
images, labels = next(iter(train_loader))
print(f"총 이미지 개수: {len(dataset)} | 클래스 종류: {dataset.classes}")
print(f"이미지 배치 크기: {images.shape} (타입: {images.dtype})")
print(f"라벨 배치 크기: {labels.shape}")


총 이미지 개수: 2148 | 클래스 종류: ['Kirmizi_Pistachio', 'Siirt_Pistachio']
이미지 배치 크기: torch.Size([4, 3, 224, 224]) (타입: torch.float32)
라벨 배치 크기: torch.Size([4])


In [4]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
print("모델 전처리 과정 : \n", weights.transforms())

모델 전처리 과정 : 
 ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.005)

In [11]:
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)
model = model.to(device)

In [12]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 50 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [13]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [14]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(val_loader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.716963  [    4/ 1718]
loss: 0.659027  [  204/ 1718]
loss: 0.669498  [  404/ 1718]
loss: 0.638610  [  604/ 1718]
loss: 0.453403  [  804/ 1718]
loss: 0.621685  [ 1004/ 1718]
loss: 0.611346  [ 1204/ 1718]
loss: 0.385962  [ 1404/ 1718]
loss: 0.294532  [ 1604/ 1718]
Test Error: 
 Accuracy: 90.2%, Avg loss: 0.536016 

Epoch 2
-------------------------------
loss: 0.287859  [    4/ 1718]
loss: 0.478605  [  204/ 1718]
loss: 0.286465  [  404/ 1718]
loss: 0.638781  [  604/ 1718]
loss: 0.173705  [  804/ 1718]
loss: 0.100869  [ 1004/ 1718]
loss: 0.677792  [ 1204/ 1718]
loss: 0.623222  [ 1404/ 1718]
loss: 0.214135  [ 1604/ 1718]
Test Error: 
 Accuracy: 44.7%, Avg loss: 0.777585 

Epoch 3
-------------------------------
loss: 0.099528  [    4/ 1718]
loss: 0.180277  [  204/ 1718]
loss: 0.153041  [  404/ 1718]
loss: 0.266111  [  604/ 1718]
loss: 0.104126  [  804/ 1718]
loss: 0.509589  [ 1004/ 1718]
loss: 0.107154  [ 1204/ 1718]
loss: 0.062356  [ 1404/ 17

In [15]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth
